In [3]:
# 修改此.ipynb的工作目录
import os
from typing import Optional

from hello_agents import HelloAgentsLLM
from sympy.printing.cxx import reserved

os.chdir(r"C:\Users\61075\PycharmProjects\learning\hello_agent")

print(os.getcwd())

C:\Users\61075\PycharmProjects\learning\hello_agent


# Memory & RAG

- 当前大模型是无状态的，每次调用都是独立的、无关联的计算，导致：上下文丢失（早期信息因为上下文窗口限制而丢失）、个性化缺失（无法记住用户偏好习惯等）、学习能力受限（无法学习过往成功或失败的经验）、一致性问题（多轮对话前后矛盾）
==> memory
- LLM知识是静态的、有限的，完全来自于训练数据，导致：知识时效性截止训练数据时间、专业领域知识不足、事实准确性、可解释性
==> RAG

MemoryTool：
- 记忆系统设计成Tool

In [ ]:
def execute(self, action: str, **kwargs) -> str:
    """
    执行记忆操作，支持：
        - add：添加记忆（4种类型：working/episodic/semantic/perceptual）
        - search：搜索记忆
        - summary：获取记忆摘要
        - stats：获取统计信息
        - update：更新记忆
        - remove：删除记忆
        - forget：遗忘记忆
        - consolidate：整合记忆（短期 -> 长期）
        - clear_all：清空所有记忆
    """

    if action == "add":
        return self._add_memory(**kwargs)
    elif action == "search":
        return self._search_memory(**kwargs)
    elif action == "summary":
        return self._get_summary(**kwargs)
    # ... 其他操作
    return "wrong action"

In [ ]:
# 操作1：add
# 会话ID自动管理（确保每个记忆都有明确的会话归属）、多模态数据智能处理（自动推断文件类型并保存相关元数据）、上下文信息自动补充（每个记忆添加时间戳和会话信息）

def _add_memory(
        self,
        content: str = None,
        memory_type: str = "working",
        importance: float = 0.5,
        file_path: str = None,
        modality: str = None,
        **metadata
) -> str:
    """添加记忆"""
    try:
        # 确保会话ID存在
        if self.current_session_id is None:
            self.current_session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        # 感知记忆（多模态）支持
        if memory_type == "perceptual" and file_path:
            inferred = modality or self._infer_modality(file_path)  # 识别具体模态类型
            metadata.setdefault("modality", inferred)
            metadata.setdefault("file_path", file_path)

        # 添加会话信息到元数据
        metadata.update({
            "session_id": self.current_session_id,
            "timestamp": datetime.now().strftime('%Y%m%d_%H%M%S'),
        })

        memory_id = self.memory_manager.add_memory(
            content=content,
            memory_type=memory_type,
            importance=importance,
            metadata=metadata,
            auto_classify=False
        )

        return f"✅ 记忆已添加 (ID: {memory_id[:8]}...)"

    except Exception as e:
        return f"❌ 添加记忆失败：{str(e)}"

In [ ]:
# 操作2：search
# 涉及语义理解、相关性计算、结果排序等

def _search_memory(
        self,
        query: str = None,
        limit: int = 5,
        memory_types: List[str] = None,
        memory_type: str = None,
        min_importance: float = 0.1,
) -> str:
    """搜索记忆"""
    try:
        # 参数标准化处理
        if memory_type and not memory_types:
            memory_types = [memory_type]

        results = self.memory_manager.retrieve_memories(
            query=query,
            limit=limit,
            memory_types=memory_types,
            min_importance=min_importance,
        )

        if not results:
            return f"🔍 未找到与 '{query}' 相关的记忆"

        # 格式化结果
        formatted_results = []
        formatted_results.append(f"🔍 找到 {len(results)} 条相关记忆:")

        for i, memory in enumerate(results):
            memory_type_label = {
                "working": "工作记忆",
                "episodic": "情景记忆",
                "semantic": "语义记忆",
                "perceptual": "感知记忆",
            }.get(memory.memory_type, memory.memory_type)

            content_preview = memory.content[:80] + "..." if len(memory.content) > 80 else memory.content
            formatted_results.append(
                f"{i}. [{memory_type_label}] {content_preview} (重要性: {memory.importance:.2f})"
            )

        return "\n".join(formatted_results)

    except Exception as e:
        return f"❌ 搜索记忆失败: {str(e)}"

In [ ]:
# 操作3：forget
# 基于重要性遗忘（删除不重要的记忆），基于时间（删除过时的记忆），基于容量（当存储接近上线时删除不重要的记忆）

def _forget_memory(
        self,
        strategy: str = "importance_based",
        threshold: float = 0.1,
        max_age_days: int = 30,
) -> str:
    """遗忘记忆"""
    try:
        count = self.memory_manger.forget_memories(
            strategy=strategy,
            threshold=threshold,
            max_age_days=max_age_days,
        )
        return f"🧹 已遗忘 {count} 条记忆（策略: {strategy}）"
    except Exception as e:
        return f"❌ 遗忘记忆失败: {str(e)}"

In [ ]:
# 操作4：consolidate
# 将重要的短期记忆working转化为长期记忆episodic,或将重要的情景记忆episodic转为语义记忆semantic

def _consolidate(
        self,
        from_type: str = "working",
        to_type: str = "episodic",
        importance_threshold: float = 0.7,
) -> str:
    """整合记忆，重要的短期记忆提升为长期记忆"""
    try:
        count = self.memory_manger.consolidate_memories(
            from_type=from_type,
            to_type=to_type,
            importance_threshold=importance_threshold,
        )
        return f"🔄 已整合 {count} 条记忆为长期记忆（{from_type} -> {to_type}，阈值={importance_threshold}）"
    except Exception as e:
        return f"❌ 整合记忆失败: {str(e)}"

MemoryManager：
- 分层设计，MemoryTool专注于用户接口和参数处理，MemoryManager负责核心的记忆管理逻辑；
- MemoryTool在初始化时会创建一个MemoryManager实例，并启用不同类型的记忆模块;
- MemoryManager负责管理不同类型的记忆模块，提供统一的接口

In [ ]:
class MemoryTool(Tool):
    """记忆工具，为agent提供记忆功能"""
    def __init__(
            self,
            user_id: str = None,
            memory_config: MemoryConfig = None,
            memory_types: List[str] = None,
    ):
        super().__init__(
            name="memory",
            description="记忆工具 - 可以存储和检索对话历史、知识和经验"
        )

        # 初始化记忆管理器
        self.memory_config = memory_config or MemoryConfig()
        self.memory_types = memory_types or ["working", "episodic", "semantic"]

        self.memory_manager = MemoryManager(
            config=self.memory_config,
            user_id=user_id,
            enable_working="working" in self.memory_types,
            enable_episodic="episodic" in self.memory_types,
            enable_semantic="semantic" in self.memory_types,
            enable_perceptual="perceptual" in self.memory_types,
        )

In [ ]:
class MemoryManager:
    """记忆管理器，统一的记忆操作接口"""
    def __init__(
            self,
            config: Optional[MemoryConfig] = None,
            user_id: str = "default_user",
            enable_working: bool = True,
            enable_episodic: bool = True,
            enable_semantic: bool = True,
            enable_perceptual: bool = False,
    ):
        self.config = config or MemoryConfig()
        self.user_id = user_id

        # 初始化存储和检索组件
        self.store = MemoryStore(self.config)
        self.retriever = MemoryRetriever(self.store, self.config)

        # 初始化各类型记忆
        self.memory_types = {}

        if enable_working:
            self.memory_types["working"] = WorkingMemory(self.config, self.store)
        if enable_episodic:
            self.memory_types["episodic"] = EpisodicMemory(self.config, self.store)
        if enable_semantic:
            self.memory_types["semantic"] = SemanticMemory(self.config, self.store)
        if enable_perceptual:
            self.memory_types["perceptual"] = PerceptualMemory(self.config, self.store)

四种记忆类型：
- WorkingMemory：工作记忆，当前对话中的临时信息，采用纯内存存储方式，配合TTL(Time to Live)机制，方便快速访问和自动清理;
- EpisodicMemory：情景记忆，具体的事件和经历，保持事件完整性和时序关系，采用SQLite+Qdrant的混合存储方案，SQLite负责结构化数据的存储和复杂查询，Qdrant负责高效和向量检索;
- SemanticMemory：语义记忆，抽象的概念、规则、知识，采用Neo4j图数据库和Qdrant向量数据库的混合架构，快速进行语义检索，并利用知识图谱进行复杂的关系推理;
- PerceptualMemory：感知记忆，支持多模态数据，采用模态分离的存储策略，避免计算时维度不匹配

In [ ]:
class WorkingMemory:
    """
    工作记忆
    - 容量有限（默认50条） + TTL自动清理
    - 纯内存存储，访问速度快
    - 混合检索，TF-IDF向量化 + 关键词匹配
    """
    def __init__(self, config: MemoryConfig):
        self.max_capacity = config.working_memory_capacity or 50
        self.max_age_minutes = config.working_memory_ttl or 60
        self.memories = []

    def add(self, memory_item: MemoryItem) -> str:
        """添加工作记忆"""
        self._expire_old_memories()     # 过期清理

        if len(self.memories) >= self.max_capacity:
            self._remove_lowest_priority_memory()   # 容量管理

        self.memories.append(memory_item)
        return memory_item.id

    def retrieve(self, query: str, limit: int = 5, **kwargs) -> List[MemoryItem]:
        """混合检索：TF-IDF向量化 + 关键词匹配"""
        self._expire_old_memories()     # 清理过期记忆

        # TF-IDF
        vector_scores = self._try_tfidf_search(query)

        # 计算综合分数
        scored_memories = []
        for memory in self.memories:
            vector_score = vector_scores.get(memory.id, 0.0)
            keyword_score = self._calculate_keyword_score(query, memory.content)

            # 混合评分
            base_relevance = vector_score * 0.7 + keyword_score * 0.3 if vector_score > 0 else keyword_score    # hybrid score
            time_decay = self._calculate_time_decay(memory.timestamp)   # 时间衰减
            importance_weight = 0.8 + (memory.importance * 0.4)     # 重要性

            final_score = base_relevance * time_decay * importance_weight
            if final_score > 0:
                scored_memories.append((final_score, memory))

        scored_memories.sort(key=lambda x: x[0], reverse=True)
        return [memory for _, memory in scored_memories[:limit]]

In [ ]:
class EpisodicMemory:
    """
    情景记忆：
    - SQLite + Qdrant混合存储架构
    - 支持时间序列和会话级检索
    - 结构化过滤 + 语义向量检索
    """
    def __init__(self, config: MemoryConfig):
        self.doc_store = SQLiteDocumentStore(config.database_path)
        self.vector_store = QdrantVectorStore(config.qdrant_url, config.qdrant_api_key)
        self.embedder = create_embedding_model_with_fallback()
        self.sessions = {}   # 会话索引，建立 session -> episode 的时间序列索引，保存会话连续性

    def add(self, memory_item: MemoryItem) -> str:
        """添加情景记忆"""
        # 创建情景对象
        episode = Episode(
            episode_id=memory_item.id,
            session_id=memory_item.metadata.get("session_id", "default"),
            timestamp=memory_item.timestamp,
            content=memory_item.content,
            context=memory_item.metadata,
        )

        # 更新会话索引
        session_id = episode.session_id
        if session_id not in self.sessions:
            self.sessions[session_id] = []
        self.sessions[session_id].append(episode.episode_id)

        # 持久化存储（SQLite + Qdrant）
        self._persist_episode(episode)
        return memory_item.id

    def retrieve(self, query: str, limit: int = 5, **kwargs) -> List[MemoryItem]:
        """混合检索：结构化过滤 + 语义向量检索"""
        # 1. 结构化过滤（时间范围、重要性等）
        candidate_ids = self._structured_filter(**kwargs)

        # 2. 向量语义检索
        hits = self._vector_search(query, limit*5, kwargs.get("user_id"))

        # 3. 综合评分与排序
        results = []
        for hit in hits:
            if self._should_include(hit, candidate_ids, kwargs):
                score = self._calculate_episode_score(hit)
                memory_item = self._create_memory_item(hit)
                results.append((score, memory_item))
        results.sort(key=lambda x: x[0], reverse=True)

        return [item for _, item in results[:limit]]

    def _calculate_episode_score(self, hit) -> float:
        """情景记忆评分算法"""
        vec_score = float(hit.get("score", 0.0))    # 向量相似度
        recency_score = self._calculate_recency(hit["metadata"]["timestamp"])   # 时间近因性
        importance = hit["metadata"].get("importance", 0.5)     # 重要性

        base_relevance = vec_score * 0.8 + recency_score * 0.2
        importance_weight = 0.8 + (importance * 0.4)

        return base_relevance * importance_weight

In [ ]:
class SemanticMemory(BaseMemory):
    """
    语义记忆：
    - huggingface中文预训练模型进行嵌入
    - 向量检索进行快速相似度匹配
    - 知识图谱存储实体和关系
    - 混合检索策略：向量 + 图 + 语义推理
    """
    def __init__(self, config: MemoryConfig, store_backend: MemoryStore):
        super().__init__(config, store_backend)

        # 嵌入模型
        self.embedding_model = get_text_embedder()

        # Qdrant数据库存储
        self.vector_store = QdrantConnectionManager.get_instance(**qdrant_config)
        self.graph_store = Neo4jGraphStore(**neo4f_config)

        # 实体和关系缓存
        self.entities: Dict[str, Entity] = {}
        self.relations: List[Relation] = []

        # NLP处理器
        self.nlp = self._init_nlp()

    def add(self, memory_item: MemoryItem) -> str:
        """添加语义记忆 - 体现知识图谱构建的完整流程"""
        # 1. 生成文本嵌入
        embedding = self.embedding_model.encode(memory_item.content)

        # 2. 提取实体和关系
        entities = self._extract_entities(memory_item.content)
        relations = self._extract_relations(memory_item.content, entities)

        # 3. 存储到Neo4j图数据库
        for entity in entities:
            self._add_entity_to_graph(entity, memory_item)
        for relation in relations:
            self._add_relation_to_graph(relation, memory_item)

        # 4. 存储到Qdrant向量数据库
        metadata = {
            "memory_id": memory_item.id,
            "entities": [e.entity_id for e in entities],
            "entities_count": len(entities),
            "relation_count": len(relations),
        }
        self.vector_store.add_vectors(
            vector=[embedding.tolist()],
            metadata=metadata,
            ids=[memory_item.id],
        )

    def retrieve(self, query: str, limit: int = 5, **kwargs) -> List[MemoryItem]:
        """检索语义记忆"""
        # 1. 向量检索
        vector_results = self._vector_search(query, limit*2, user_id)

        # 2. 图检索
        graph_results = self._graph_search(query, limit*2, user_id)

        # 3. 混合排序
        combined_results = self._combine_and_rank_results(
            vector_results, graph_results, query, limit
        )

        return combined_results[:limit]

    def _combine_and_rank_results(self, vector_results, graph_results, query, limit):
        """混合排序结果"""
        combined = {}

        # 合并向量和图检索结果
        for result in vector_results:
            combined[result["memory_id"]] = {
                **result,
                "vector_score": result.get("score", 0.0),
                "graph_score": 0.0
            }

        for result in graph_results:
            memory_id = result["memory_id"]
            if memory_id in combined:
                combined[memory_id]["graph_score"] = result.get("similarity", 0.0)
            else:
                combined[memory_id] = {
                    **result,
                    "vector_score": 0.0,
                    "graph_score": result.get("similarity", 0.0)
                }

        # 计算混合分数
        for memory_id, result in combined.items():
            vector_score = result["vector_score"]
            graph_score = result["graph_score"]
            importance = result.get("importance", 0.5)

            base_relevance = vector_score * 0.7 + graph_score * 0.3
            importance_weight = 0.8 + (importance * 0.4)
            combined_score = base_relevance * importance_weight
            result["combined_score"] = combined_score

        # 排序并返回
        sorted_results = sorted(
            combined.values(), key=lambda k: k["combined_score"], reverse=True
        )
        return  sorted_results

In [ ]:
class PerceptualMemory(BaseMemory):
    """
    感知记忆实现
    - 支持多模态数据
    - 跨模态相似度搜索
    - 感知数据的语义理解
    - 支持内容生成和检索
    """
    def __init__(self, config: MemoryConfig, store_backend: MemoryStore):
        super().__init__(config, store_backend)

        # 多模态编码器
        self.text_embedder = get_text_embedder()
        self._clip_model = self._init_clip_model()  # 图像编码
        self._clap_model = self._init_clap_model()  # 音频编码

        # 按模态分离的向量存储
        self.vector_stores = {
            "text": QdrantConnectionManager.get_instance(
                collection_name="perceptual_text",
                vector_size=self.vector_dim,
            ),
            "image": QdrantConnectionManager.get_instance(
                collection_name="perceptual_image",
                vector_size=self._image_dim,
            ),
            "audio": QdrantConnectionManager.get_instance(
                collection_name="perceptual_audio",
                vector_size=self._audio_dim
            )
        }

    def retrieve(self, query: str, limit: int = 5, **kwargs) -> List[MemoryItem]:
        """检索感知记忆：可筛选模态；同模态向量检索+时间/重要性融合"""
        user_id = kwargs.get("user_id")
        target_modality = kwargs.get("target_modality")
        query_modality = kwargs.get("query_modality", target_modality or "text")

        # 同模态向量检索
        try:
            query_vector = self._encode_data(query, query_modality)
            store = self._get_vector_store_for_modality(target_modality or query_modality)

            where = {"memory_type": "perceptual"}
            if user_id:
                where["user_id"] = user_id
            if target_modality:
                where["target_modality"] = target_modality

            hits = store.search_similar(
                query_vector=query_vector,
                limit=max(limit*5, 20),
                where=where,
            )
        except Exception:
            hits = []

        # 融合排序：向量相似度 + 时间近因性 + 重要性权重
        result = []
        for hit in hits:
            vector_score = float(hit.get("score", 0.0))
            recency_score = self._calculate_episode_score(hit["metadata"]["timestamp"])
            importance = hit["metadata"].get("importance", 0.5)

            base_relevance = vector_score * 0.8 + recency_score * 0.2
            importance_weight = 0.8 + (importance * 0.4)
            combined_score = base_relevance * importance_weight

            result.append((combined_score, self._create_memory_item(hit)))

        results.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in results[:limit]]

    @staticmethod
    def _calculate_recency_score(timestamp: str) -> float:
        """计算时间近因性得分"""
        try:
            memory_time = datetime.fromisoformat(timestamp)
            current_time = datetime.now()
            age_hours = (current_time - memory_time).total_seconds() / 3600

            # 指数衰减：24小时内保持高分，之后逐渐衰减
            decay_factor = 0.1  # 衰减指数
            recency_score = math.exp(-decay_factor * age_hours / 24)
            return max(0.1, recency_score)
        except Exception:
            return 0.5

## RAG：
- 数据处理流程：处理和存储知识文档，将一切传入的外部知识统一转化为Markdown格式；
- 查询与生成流程：根据查询检索相关信息并生成回答

---

将RAG设计成工具，其pipeline：

用户层：RAGTool统一接口

  ↓

应用层：智能问答、搜索、管理

  ↓

处理层：文档解析、分块、向量化

  ↓

存储层：向量数据库、文档存储

  ↓

基础层：嵌入模型、LLM、数据库

In [ ]:
class RAGTool(Tool):
    """
    RAG工具：
    - 添加多格式文档（pdf、office、图片、音频等）
    - 智能检索与召回
    - llm增强问答
    - 知识库管理
    """
    def __init__(
            self,
            knowledge_base_path: str = "./knowledge_base",
            qdrant_url: str = None,
            qdrant_api_key: str = None,
            collection_name: str = "rag_knowledge_base",
            rag_namespace: str = "default"
    ):
        # 初始化rag pipeline
        self._pipelines: Dict[str, Dict[str, Any]] = {}
        self.llm = HelloAgentsLLM()

        # 创建默认pipeline
        default_pipeline = create_rag_pipeline(
            qdrant_url=self.qdrant_url,
            qdrant_api_key=self.qdrant_api_key,
            collection_name=self.collection_name,
            rag_namespace=self.rag_namespace,
        )
        self._pipelines[self.rag_namespace] = default_pipeline

任意格式文档 → MarkItDown转换 → Markdown文本 → 智能分块 → 向量化 → 存储检索

In [ ]:
def _convert_to_markdown(path: str) -> str:
    """
    使用 markitdown 将任意格式文档转换为 markdown 文本
    支持格式：
    - 文档：pdf、word、excel、powerpoint
    - 图像：jpg、png、gif
    - 音频：MP3、mav、m4a
    - 文本：txt、csv、json、xml、html
    - 代码：python、JavaScript、Java等
    """
    if not os.path.exists(path):
        return ""

    # 对pdf文件使用增强处理
    ext = (os.path.splitext(path)[1] or "").lower()
    if ext == "pdf":
        return _enhanced_pdf_processing(path)

    # 其他格式使用markitdown统一转换
    md_instance = _get_markitdown_instance()
    if md_instance is None:
        return _fallback_text_reader(path)

    try:
        result = md_instance.convert(path)
        markdown_text = getattr(result, "text_content", None)
        if isinstance(markdown_text, str) and markdown_text.strip():
            print(f"[RAG] MarkItDown转化成功：{path} -> {len(markdown_text)} chars Markdown")
            return markdown_text
        return ""
    except Exception as e:
        print(f"[WARNING] MarkItDown转化失败 {path}: {e}")
        return _fallback_text_reader(path)

MarkDown结构的分块流程：

标准markdown文本：统一格式、结构清晰

↓

标题层次清晰： #/##/### 层次识别

↓

段落语义分割：语义边界完整

↓

token计算分块：控制大小，优化检索

↓

重叠策略优化：信息连续性、保持上下文

↓

向量化准备：嵌入向量，相似度匹配

In [ ]:
def _split_paragraphs_with_headings(text: str) -> List[str]:
    """根据标题层次分割段落，保持语义完整性"""
    lines = text.splitlines()   # 根据换行符拆分字符串，输出为列表
    heading_stack: List[str] = []   # 标题层级路径，保留文档语义结构，方便检索
    paragraphs: List[Dict] = []
    buf: List[str] = []
    char_pos = 0

    def flush_buf(end_pos: int):
        """把缓冲区buf中累计的文本行整理成结构化的paragraphs段落文本"""
        if not buf:
            return
        content = "\n".join(buf).strip()
        if not content:
            return
        paragraphs.append({
            "content": content,
            "heading_path": " > ".join(heading_stack) if heading_stack else None,
            "start": max(0, end_pos - len(content)),
            "end": end_pos,
        })

    for ln in lines:
        raw = ln
        if raw.strip().startswith("#"):
            # 处理标题行
            flush_buf(char_pos)
            level = len(raw) - len(raw.lstrip("#"))     # 提取“#”个数，用于识别标题层级
            title = raw.lstrip("#").strip()

            if level <= 0:  # 防御性判断
                level = 1
            if level <= len(heading_stack):
                heading_stack = heading_stack[:level-1]
            heading_stack.append(title)

            char_pos += len(raw) + 1
            continue

        # 段落内容累积
        if raw.strip() == "":
            flush_buf(char_pos)
            buf = []
        else:
            buf.append(raw)
        char_pos += len(raw) + 1

    flush_buf(char_pos)

    if not paragraphs:
        paragraphs = [{
            "content": text,
            "heading_path": None,
            "start": 0,
            "end": len(text),
        }]

    return paragraphs

In [ ]:
def _chunk_paragraphs(paragraphs: List[Dict], chunk_tokens: int, overlap_tokens: int) -> List[Dict]:
    """在段落分割的基础上，基于token数量的智能分块"""
    chunks: List[Dict] = []
    cur: List[Dict] = []
    cur_tokens = 0
    i = 0

    while i < len(paragraphs):
        p = paragraphs[i]
        p_tokens = _approx_token_len(p["content"]) or 1

        if cur_tokens + p_tokens <= chunk_tokens or not cur:
            # 把多个 paragraphs 合并成适合 embedding 的 chunk
            cur.append(p)
            cur_tokens += p_tokens
            i += 1
        else:
            # 生成当前chunk
            content = "\n\n".join(x["content"] for x in cur)
            start = cur[0]["start"]
            end = cur[-1]["end"]
            heading_path = next((x["heading_path"] for x in reversed(cur) if x.get("heading_path")), None)      # 找 chunk中最近的heading_path

            chunks.append({
                "content": content,
                "start": start,
                "end": end,
                "heading_path": heading_path,
            })

            # 构建重叠部分
            if overlap_tokens > 0 and cur:
                kept: List[str] = []
                kept_tokens = 0
                for x in reversed(cur):     # 从chunk尾部开始倒着保留
                    t = _approx_token_len(x["content"]) or 1
                    if kept_tokens + t > overlap_tokens:
                        break
                    kept.append(x)
                    kept_tokens += t
                cur = list(reversed(cur))   # 作为下一个chunk的开头
                cur_tokens = kept_tokens
            else:
                cur = []
                cur_tokens = 0

    # 处理最后一个分块
    if cur:
        content = "\n\n".join(x["content"] for x in cur)
        start = cur[0]["start"]
        end = cur[-1]["end"]
        heading_path = next((x["heading_path"] for x in cur if x.get("heading_path")), None)

        chunks.append({
            "content": content,
            "start": start,
            "end": end,
            "heading_path": heading_path,
        })

    return chunks

In [ ]:
def _approx_token_len(text: str) -> int:
    """近似估计token长度，支持中英混合"""
    # CJK字符按照 1 token计算
    cjk = sum(1 for ch in text if _is_cjk(ch))
    # 其他字符按空白分词计算
    non_cjk_tokens = len([t for t in text.split() if t])
    return cjk + non_cjk_tokens


def _is_cjk(ch: str) -> bool:
    """判断是否为CJK字符"""
    code = ord(ch)
    return (
        0x4E00 <= code <= 0x9FFF or     # CJK统一汉字
        0x3400 <= code <= 0x4DBF or     # CJK扩展A
        0x20000 <= code <= 0x2A6DF or   # CJK扩展B
        0x2A700 <= code <= 0x2B73F or   # CJK扩展C
        0x2B740 <= code <= 0x2B81F or   # CJK扩展D
        0x2B820 <= code <= 0x2CEAF or   # CJK扩展E
        0xF900 <= code <= 0xFAFF        # CJK兼容汉字
    )

In [ ]:
def index_chunk(
        store = None,
        chunks: List[Dict] = None,
        cache_db: Optional[str] = None,
        batch_size: int = 64,
        rag_namespace: str = "default"
) -> None:
    """
    使用统一嵌入和 Qdrant 存储对 Markdown chunks进行索引
    """
    if not chunks:
        print("[RAG] No chunks to index")
        return

    # 使用统一嵌入模型
    embedder = get_text_embedder()
    dimension = get_dimension(384)

    # 创建默认Qdrant存储
    if store is None:
        store = _create_default_vector_store(dimension)
        print(f"[RAG] Created default Qdrant store with dimension {dimension}")

    # 预处理markdown文本以获得更好的嵌入质量
    processed_texts = []
    for c in chunks:
        raw_content = c["content"]
        processed_content = _preprocess_markdown_for_embedding(raw_content)
        processed_texts.append(processed_content)
    print(f"[RAG] Embedding start: total_texts={len(processed_texts)} batch_size={batch_size}")

    # 批量编码
    vecs: List[List[float]] = []
    for i in range(0, len(processed_texts), batch_size):
        part = processed_texts[i:i + batch_size]
        try:
            # 使用统一嵌入器，内部处理缓存
            part_vecs = embedder.encode(part)

            # 标准化为 List[List[float]] batch格式
            if not isinstance(part_vecs, list):
                if hasattr(part_vecs, "tolist"):
                    part_vecs = [part_vecs.tolist()]
                else:
                    part_vecs = [list(part_vecs)]

            # 处理向量格式和维度
            for v in part_vecs:
                try:
                    if hasattr(v, "tolist"):
                        v = v.tolist()
                    v_norm = [float(x) for x in v]

                    # 维度检查和调整(VectorDB 要求所有向量维度固定)
                    if len(v_norm) != dimension:
                        print(f"[WARNING] 向量维度异常：期望{dimension}, 实际{len(v_norm)}")
                        if len(v_norm) < dimension:     # 如果维度不足 -> padding
                            v_norm.extend([0.0] * (dimension - len(v_norm)))
                        else:   # 如果维度过长 -> 截断
                            v_norm = v_norm[:dimension]
                    vecs.append(v_norm)

                except Exception as e:
                    print(f"[WARNING] 向量转换失败: {e}, 使用零向量")
                    vecs.append([0.0] * dimension)

        except Exception as e:
            print(f"[WARNING] Batch {i} encoding failed: {e}")
            # 实现重试机制
            pass

        print(f"[RAG] Embedding progress: {min(i+batch_size, len(processed_texts))}/{len(processed_texts)}")

检索策略：
- 多查询扩展：Multi-Query Expansion, MQE. 通过生成语义等价的多样化查询来提高检索召回率的技术。
    - 同一个问题可以有多种不同的表述方式，而不同的表述可能匹配到不同的相关文档，通过并行执行这些扩展查询并合并结果，覆盖更广泛的相关文档
    - 优势在于，能够自动理解用户查询的多种可能含义，特别是对于模糊查询或专业术语查询效果好
- 假设文档嵌入：Hypothetical Document Embeddings, HyDE. 用答案找答案。
    - 让LLM先生成一个假设性的答案段落，然后用这个答案段落去检索真实文档，从而缩小查询和文档之间的语义鸿沟
    - 优势在于，假设答案与真实答案的语义空间更接近
- 扩展检索框架：将MQE和HyDE整合

In [ ]:
def _prompt_mqe(query: str, n: int) -> List[str]:
    """使用LLM生成多样化的查询扩展"""
    try:
        from ...core.llm import HelloAgentsLLM
        llm = HelloAgentsLLM()
        prompt = [
            {"role": "system", "content": "你是检索查询扩展助手。生成语义等价或互补的多样化查询。使用中文，简短，避免标点。"},
            {"role": "user", "content": f"原始查询：{query}\n请给出{n}个不同表述的查询，每行一个。"}
        ]
        text = llm.invoke(prompt)
        lines = [ln.strip("- \t") for ln in (text or "").splitlines()]
        outs = [ln for ln in lines if ln]
        return outs[:n] or [query]
    except Exception as e:
        return [query]

In [ ]:
def _prompt_hyde(query: str) -> Optional[str]:
    """生成假设性文档用于改善检索"""
    try:
        from ...core.hyde import HydeAgents
        llm = HelloAgentsLLM()
        prompt = [
            {"role": "system", "content": "根据用户问题，先写一段可能的答案性段落，用于向量检索的查询文档（不要分析过程）。"},
            {"role": "user", "content": f"问题：{query}\n请直接写一段中等长度、客观、包含关键术语的段落。"}
        ]
        return llm.invoke(prompt)
    except Exception as e:
        return None

In [ ]:
def search_vectors_expanded(
        store: None,
        query: str = "",
        top_k: int =8,
        rag_namespace: Optional[str] = None,
        only_rag_data: bool = True,
        score_threshold: Optional[float] = None,
        enable_mqe: bool = False,
        mqe_expansions: int = 2,
        enable_hyde: bool = False,
        candidate_pool_multiplier: int = 4,
) -> List[Dict]:
    """
    Search with query expansion using unified embedding and Qdrant.
    """
    if not query:
        return []

    # 创建默认存储
    if store is None:
        store = _create_default_vector_store()

    # 查询扩展
    expansions: List[str] = [query]

    if enable_mqe and mqe_expansions > 0:
        expansions.extend(_prompt_mqe(query, mqe_expansions))
    if enable_hyde:
        hyde_text = _prompt_hyde(query)
        if hyde_text:
            expansions.extend(hyde_text)

    # 去重和修剪
    uniq: List[str] = []
    for e in expansions:
        if e and e not in uniq:
            uniq.append(e)
    expansions = uniq

    # 分配候选池
    pool = max(top_k * candidate_pool_multiplier, 20)   # 总候选池大小
    per = max(1, pool // max(1, len(expansions)))   # 每个expansion分配多少候选（均分）

    # 构建rag数据过滤器
    where = {'memory_type': 'rag_chunk'}
    if only_rag_data:
        where['is_rag_data'] = True
        where['data_source'] = 'rag_pipeline'
    if rag_namespace:
        where['rag_namespace'] = rag_namespace

    # 收集所有扩展查询的结果
    agg: Dict[str, Dict] = {}
    for q in expansions:
        qv = embed_query(q)
        hits = store.search_similar(
            query_vectors=qv,
            limit=per,
            score_threshold=score_threshold,
            where=where,
        )
        for h in hits:
            mid = h.get("metadata", {}).get("memory_id", h.get("id"))
            s = float(h.get("score", 0.0))
            if mid not in agg or s >float(agg[mid].get("score", 0.0)):
                agg[mid] = h

    # 按分数排序返回
    merged = list(agg.values())
    merged.sort(key=lambda x: float(x.get("score", 0.0)), reverse=True)
    return merged[:top_k]

在 06_Memory 项目中，学习实现记忆系统和RAG系统：

1. 文档处理：使用MarkItDown将PDF转换为MarkDown，然后进行智能分块，向量化和构建索引；
2. 检索问答：MQE提高召回率，HyDE改善检索精度；
3. 记忆管理：工作记忆管理当前学习任务和上下文，情景记忆记录学习事件和查询历史，语义系统存储概念知识和理解，感知记忆处理文档特征和多模态信息；
4. 个性化：基于学习历史的个性化推荐，记忆整合和选择性遗忘，学习报告生成和进度追踪